In [ ]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

from tqdm import tqdm
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

print(f"Project root added to sys.path: {PROJECT_ROOT}")

In [ ]:
from prediction_tennis.src.utils.file_utils import load_csv_files_from_directory, combine_dataframes, _save_dataframe_to_csv

In [ ]:
# Define a variable to choose between "hard" or "clay"
SELECTED_SURFACE = "all"  


## 1. Load the data

In [ ]:
# Base raw data directory
DATA_FOLDER = Path("../../../data")

# Subdirectories for raw and processed data
DATA_FOLDER_RAW = DATA_FOLDER / "01_raw"
DATA_FOLDER_PROCESSED = DATA_FOLDER / "02_processed"

# Subdirectories for specific raw sources
FOLDER_FLASHSCORE_RAW = DATA_FOLDER_RAW / "flashscore"
FOLDER_WIKIPEDIA_RAW = DATA_FOLDER_RAW / "wikipedia"
FOLDER_ATPTOUR_RAW = DATA_FOLDER_RAW / "atptour"

# Specific files
FILE_ATPTOUR_PLAYERS = FOLDER_ATPTOUR_RAW / "players.csv"

# Processed file (depends on selected surface)
FILE_CLEANING = DATA_FOLDER_PROCESSED / f"cleaning__processed_value__{SELECTED_SURFACE}.csv"

In [ ]:
# Load Flashscore data
flashscore_dataframes  = load_csv_files_from_directory(directory_path= FOLDER_FLASHSCORE_RAW, pattern= "tournament_")
combined_flashscore_df = combine_dataframes(flashscore_dataframes)

# Load Wikipedia data
wikipedia_dataframes = load_csv_files_from_directory(FOLDER_WIKIPEDIA_RAW, "tournament")
combined_wikipedia_df = combine_dataframes(wikipedia_dataframes)

# Load ATP Tour data
atptour_df = pd.read_csv(FILE_ATPTOUR_PLAYERS)


In [ ]:
print(" --- ")

print(f'[FLASHCORE] len : {len(combined_flashscore_df)}')
print(f'[FLASHCORE] len columns : {len(combined_flashscore_df.columns)}')

print(" --- ")

print(f'[WIKIPEDIA] len : {len(combined_wikipedia_df)}')
print(f'[WIKIPEDIA] len columns : {len(combined_wikipedia_df.columns)}')

print(" --- ")

print(f'[ATP TOUR] len : {len(atptour_df)}')
print(f'[ATP TOUR] len columns : {len(atptour_df.columns)}')

print(" --- ")

### 1.1 Partiular case

In [ ]:
# Adjust year for "Olympic Games" in 2021 to 2020 due to COVID-19
combined_wikipedia_df.loc[(combined_wikipedia_df['name'] == 'olympic-games') & (combined_wikipedia_df['year'] == 2021), 'year'] = 2020

## 2. Similarity metric

### 2.a Link Flashscore and wikipedia

In [ ]:
combined_flashscore_df["tournament_year"].astype(int)

list_tournament_flashcore = combined_flashscore_df[combined_flashscore_df["tournament_year"]>=1998]["tournament_slug"].unique().tolist()
list_tournament_wikipedia = combined_wikipedia_df["name"].unique().tolist()

print(f"flashscore : {len(list_tournament_flashcore)}")
print(f"wikipedia : {len(list_tournament_wikipedia)}")


In [ ]:
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Define a function to match tournaments between sources
def match_tournaments(flashcore_list, wikipedia_list, threshold=80):
    matched_tournaments = []
    
    for flashcore_tournament in flashcore_list:
        best_match = process.extractOne(flashcore_tournament, wikipedia_list, scorer=fuzz.token_sort_ratio)
        
        # Check if the best match meets the threshold
        if best_match[1] >= threshold:
            matched_tournaments.append((flashcore_tournament, best_match[0]))
        else:
            print(f"No match found for: {flashcore_tournament} => best match: {best_match}")
    
    return matched_tournaments

# Get the matched tournaments
matches = match_tournaments(list_tournament_flashcore, list_tournament_wikipedia)


In [ ]:
combined_df = combined_flashscore_df.copy()

In [ ]:
for column in ['type', 'money', 'surface']:
    if column not in combined_df.columns:
        combined_df[column] = None

for flashscore_tournament, wikipedia_tournament in tqdm(matches):
    # Filter rows for the current Flashscore tournament.
    flashscore_rows = combined_df[combined_df["tournament_slug"] == flashscore_tournament]
    
    for index, row in flashscore_rows.iterrows():
        tournament_year = row["tournament_year"]
        
        # Filter Wikipedia DataFrame based on tournament name and year.
        wikipedia_match = combined_wikipedia_df[
            (combined_wikipedia_df["name"] == wikipedia_tournament) &
            (combined_wikipedia_df["year"] == tournament_year)
        ]
        
        if not wikipedia_match.empty:
            # Use the first match if multiple rows are returned.
            wikipedia_row = wikipedia_match.iloc[0]
            
            # Update the corresponding Flashscore row with values from Wikipedia.
            combined_df.loc[index, "type"] = wikipedia_row["type"]
            combined_df.loc[index, "money"] = wikipedia_row["money"]
            combined_df.loc[index, "surface"] = wikipedia_row["surface"]


In [ ]:
min_positive_money = combined_df.loc[combined_df["money"] > 100_000, "money"].min()
combined_df.loc[combined_df["money"] == 0, "money"] = min_positive_money
print(f"0$ replace by : {min_positive_money} $")

In [ ]:
total_rows = len(combined_df[combined_df["tournament_year"]>=1998])
none_count = combined_df[combined_df["tournament_year"]>=1998]['type'].isna().sum()

none_percentage = (none_count / total_rows) * 100 if total_rows > 0 else 0.0

print(f"Percentage of rows with missing values: {none_percentage:.2f}%")
# 0.88%

In [ ]:
combined_df["tournament_year"].astype(int)
combined_df[(combined_df['type'].isna()) & (combined_df["tournament_year"]>1998) ][["tournament_slug", "tournament_year"]].drop_duplicates()

In [ ]:
for flashscore_tournament, wikipedia_tournament in matches:
    if flashscore_tournament == "brighton":
        print(f"flashscore_tournament : {flashscore_tournament}")
        print(f"wikipedia_tournament : {wikipedia_tournament}")

In [ ]:
combined_wikipedia_df[combined_wikipedia_df["name"] == " brighton"]

In [ ]:
list_cotes = []

cotes_list = ["Unibet", "Betclic", "Betfair", "Bwin", "Netbet", "Parions-Sport"]
for cote in cotes_list:

    p1_odd_home_away = f"p1_odd_home_away_{cote}_full_start"
    p2_odd_home_away = f"p2_odd_home_away_{cote}_full_start"
    p1_odd_home_away_end = f"p1_odd_home_away_{cote}_full_end"
    p2_odd_home_away_end = f"p2_odd_home_away_{cote}_full_end"

    list_cotes.append(p1_odd_home_away)
    list_cotes.append(p2_odd_home_away)
    list_cotes.append(p1_odd_home_away_end)
    list_cotes.append(p2_odd_home_away_end)

available_cotes = [col for col in list_cotes if col in combined_df.columns]
available_cotes

### 2.b Combine AtpTour with Flashscore and wikipedia

In [ ]:

import numpy as np


combined_df[["height_p1","height_p2"]] = None

combined_df["match_date"] = pd.to_datetime(combined_df["match_date"], errors='coerce')


for index, player_info in tqdm(atptour_df.iterrows(), total=len(atptour_df), desc="Enriching player features"):
    player_id = player_info["id"]
    birthday = pd.to_datetime(player_info["birthday"], errors="coerce")
    
    # Convert turn_pro to a datetime at start of that year for precise calculation
    try:
        turn_pro_year = int(player_info.get("turn_pro", None))
        turn_pro_date = pd.Timestamp(year=turn_pro_year, month=1, day=1)
    except (ValueError, TypeError):
        turn_pro_date = pd.NaT
    
    for p_num in ['1', '2']:
        mask = combined_df[f"player{p_num}_id"] == player_id
        if not mask.any():
            continue
        
        combined_df.loc[mask, f"height_p{p_num}"]  = player_info.get("height", np.nan)
        combined_df.loc[mask, f"birthday_p{p_num}"] = birthday
        combined_df.loc[mask, f"turn_pro_p{p_num}"] = turn_pro_year
        combined_df.loc[mask, f"hand_p{p_num}"]     = player_info.get("hand", np.nan)
        combined_df.loc[mask, f"backhand_p{p_num}"] = player_info.get("backhand", np.nan)

        match_dates = combined_df.loc[mask, "match_date"]

        # Calculate age at match time (in years)
        if pd.notna(birthday):
            ages = match_dates.apply(lambda md: ((md - birthday).days / 365.25) if pd.notna(md) else np.nan).astype(float)
            combined_df.loc[mask, f"age_p{p_num}"] = ages
        else:
            combined_df.loc[mask, f"age_p{p_num}"] = np.nan

        # Calculate years since turning pro at match time (in years, float)
        if pd.notna(turn_pro_date):
            years_since_pro = match_dates.apply(lambda md: ((md - turn_pro_date).days / 365.25) if pd.notna(md) else np.nan).astype(float)
            combined_df.loc[mask, f"years_since_turn_pro_p{p_num}"] = years_since_pro
        else:
            combined_df.loc[mask, f"years_since_turn_pro_p{p_num}"] = np.nan


In [ ]:
# Keep only rows where both players are at least 13
combined_df = combined_df[(combined_df["age_p1"] >= 13) & (combined_df["age_p2"] >= 13)]

In [ ]:
test_columns = ["age_p1","age_p2",
                "player1_id","player2_id",
                "height_p1","height_p2",
                "turn_pro_p1","turn_pro_p2",
                "years_since_turn_pro_p1","years_since_turn_pro_p2",
                "hand_p1","hand_p2",
                "backhand_p1","backhand_p2"]

nb_nan = combined_df[test_columns].isna().any(axis=1).sum()
print(f"nb nan value : {nb_nan}")


## 3. Select columns to keep

In [ ]:
# Keep all columns
df = combined_df.copy()

In [ ]:
df.info()

## 4. Remove duplicate rows

In [ ]:
df.drop_duplicates(inplace=True)
df.drop_duplicates(subset=['match_id', 'player1_id', 'player2_id'], inplace=True)

print(f'len : {len(df)}')
print(" --- ")
print(f'len columns : {len(df.columns)}')
print(" --- ")
df.info()


## 5. Handle missing values

In [ ]:
# Display missing value counts for reference
print("Missing values before cleaning:\n", df.isnull().sum())

## 6. Convert string IDs for tournament

In [ ]:
# Use pd.factorize individually (since we aren't combining these)
df['tournament_id'] = pd.factorize(df['tournament_id'])[0]

print(f'len : {len(df)}')
print(" --- ")
print(f'len columns : {len(df.columns)}')
print(" --- ")
df.info()

## 7. Combined Factorization for Player IDs

In [ ]:
# Combine player1_id and player2_id to create a universal mapping
combined_ids = pd.concat([df['player1_id'], df['player2_id']])
labels_ids, uniques_ids = pd.factorize(combined_ids)

# Save mapping (uniques_ids) for reverse lookup if needed:
player_id_mapping = {code: val for code, val in enumerate(uniques_ids)}

# Apply the mapping consistently to both columns
id_map = {val: code for code, val in enumerate(uniques_ids)}
df['player1_id_factor'] = df['player1_id'].map(id_map)
df['player2_id_factor'] = df['player2_id'].map(id_map)

In [ ]:
print(f'len : {len(df)}')
print(" --- ")
print(f'len columns : {len(df.columns)}')
print(" --- ")
df.info()

## 8. Combined Factorization for Nationalities

In [ ]:
# Factorize the union of nationalities from both players
combined_nats = pd.concat([df['player1_nationality'], df['player2_nationality']])
labels_nats, uniques_nats = pd.factorize(combined_nats)

# Save mapping for potential reverse lookup:
nationality_mapping = {code: val for code, val in enumerate(uniques_nats)}

# Apply the mapping consistently to both columns
nat_map = {val: code for code, val in enumerate(uniques_nats)}
df['player1_nationality_factor'] = df['player1_nationality'].map(nat_map)
df['player2_nationality_factor'] = df['player2_nationality'].map(nat_map)

In [ ]:
print(f'len : {len(df)}')
print(" --- ")
print(f'len columns : {len(df.columns)}')
print(" --- ")
df.info()

## 9. Map 'round' column to numeric values

In [ ]:
df['round'].value_counts() / len(df) * 100

In [ ]:
round_mapping = {
    "final"         : 1,
    "semi_finals"   : 2,
    "robin"         : 3,
    "quarter_finals": 4,
    "round_of_8"    : 8,
    "round_of_16"   : 16,
    "round_of_32"   : 32,
    "round_of_64"   : 64,
    "NOT Play Off"  : 128,
    "qualif"        : 128,
}

df['round'] = df['round'].map(round_mapping)

In [ ]:
print(f'len : {len(df)}')
print(f'len columns : {len(df.columns)}')
df.info()

## 10. Map 'type' column to numeric values

In [ ]:
df['type'].value_counts() / len(df) * 100

In [ ]:
type_mapping = {
    "ATP 250"     : 250,
    "ATP 500"     : 500,
    "ATP 1000"    : 1000,
    "Masters Cup" : 1500,# Assuming it's like ATP Finals, usually 1500 max
    "Grand Chelem": 2000,
    "JO"          : 750  # Olympics, could be another value depending on ranking weight
}

df['type'] = df['type'].map(type_mapping)

## 11. Map 'Surface' column to numeric values

In [ ]:
df['surface'].value_counts() / len(df) * 100

In [ ]:
surface_mapping = {
    "hard (extern)"  : "hard",
    "hard (intern)"  : "hard",
    "clay (extern)"  : "clay",
    "clay (intern)"  : "clay",
    "grass (extern)" : "grass",  
    "carpet (intern)": "carpet",  
    "carpet (extern)": "carpet"
}
df['surface_group'] = df['surface'].map(surface_mapping)

# Filter the dataset based on the selected surface
if SELECTED_SURFACE == "hard":
    df_selected = df[df['surface_group'] == "hard"]
elif SELECTED_SURFACE == "clay":
    df_selected = df[df['surface_group'] == "clay"]
elif SELECTED_SURFACE == "all":
    df_selected = df
else:
    raise NotImplementedError(f"{SELECTED_SURFACE} not implemente ")

# Drop the intermediate column if not needed
# df_selected = df_selected.drop(columns=['surface_group'])

# Reset index
df = df_selected.reset_index(drop=True)

## 12. Hand value to binary 

In [ ]:
# Mapping dictionary
HAND_MAP = {
    "Right": 1,
    "Left": 0,
    "X": -1  # Consider np.nan if you want to impute missing later
}

# Mapping dictionary
BACKHAND_MAP = {
    "two_handed": 1,
    "one_handed": 0,
}

In [ ]:
df["hand_p1_encoded"] = df["hand_p1"].map(HAND_MAP)
df["hand_p2_encoded"] = df["hand_p2"].map(HAND_MAP)

df["backhand_p1_encoded"] = df["backhand_p1"].map(BACKHAND_MAP)
df["backhand_p2_encoded"] = df["backhand_p2"].map(BACKHAND_MAP)

## 13. Height Mean value 

In [ ]:
combined_df["height_p1"] = pd.to_numeric(combined_df["height_p1"], errors="coerce")
combined_df["height_p2"] = pd.to_numeric(combined_df["height_p2"], errors="coerce")

# Compute the mean height ignoring NaN
mean_height = np.nanmean(combined_df[["height_p1", "height_p2"]].values.ravel())

print(f"Mean height (ignoring NaN): {mean_height:.2f}")

# Replace NaN values with the mean
combined_df[["height_p1", "height_p2"]] = combined_df[["height_p1", "height_p2"]].fillna(mean_height)


In [ ]:
combined_df[["match_id", "height_p1", "height_p2"]].info()

## 14. Keep only rows where 'status' is "Finish"

In [ ]:
df = df[df['status'] == 'FINISH']
df = df[df['winner'].isin([1, 2])]

print(f'len : {len(df)}')
print(" --- ")
print(f'len columns : {len(df.columns)}')
print(" --- ")
df.info()

In [ ]:
df['winner'].value_counts() / len(df) * 100

## 15. Convert 'timestamp' to datetime and then to integer (epoch seconds)

In [ ]:
df['timestamp'] = df['timestamp'].view('int64')
df['timestamp_proccesed'] = df['timestamp'].view('int64') / 10**9

## 16. Sort DataFrame by timestamp

In [ ]:
df.sort_values('timestamp_proccesed', inplace=True)
df.head(10)

## 17. Reset index after cleaning

In [ ]:
df.reset_index(drop=True, inplace=True)

## 18. Summary of saved mappings for reverse lookup

In [ ]:
# These dictionaries can be used to reverse-map factorized values:
# For player IDs:
print("Player ID Mapping (int -> original):", player_id_mapping)

# For nationalities:
print("Nationality Mapping (int -> original):", nationality_mapping)

df.info()

In [ ]:
_save_dataframe_to_csv(dataframe=df, csv_file_path=FILE_CLEANING)